# Complete Retail Demand Forecasting Project

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ericmavigo/retail-demand-forecasting/blob/main/notebooks/00_RUN_COMPLETE_PROJECT_IN_COLAB.ipynb)

**Goal:** turn five years of M5 retail history into business insight, a leakage-free 28-day forecast and inventory decisions. Every transformation is visible in this notebook; no external Python scripts are required.

> **Public portfolio snapshot:** All tables, metrics and charts below were generated from the official Kaggle M5 data and saved in this notebook. You can review the complete work without credentials. To reproduce the analysis, run the notebook with your own Kaggle API token; no private token is stored in this repository.

## 1. Environment
Run every cell from top to bottom. In Colab, the first cell installs the required packages.

In [1]:
import sys, subprocess
from pathlib import Path

if 'google.colab' in sys.modules:
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        'kagglehub>=1.0,<2', 'pandas>=2.2,<3', 'numpy>=2,<3',
        'plotly>=5.24,<7', 'scikit-learn>=1.5,<2', 'lightgbm>=4.5,<5'
    ])

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
pd.set_option('display.max_columns', 30)


## Kaggle authentication and official data

Before running this cell:

1. Open the [M5 competition page](https://www.kaggle.com/competitions/m5-forecasting-accuracy/rules) and accept the rules.
2. Create an API token in [Kaggle Settings](https://www.kaggle.com/settings/api).
3. Recommended for Colab: open the **Secrets** panel, add a secret named `KAGGLE_API_TOKEN`, paste the token as its value and enable notebook access.

If the Colab secret is unavailable, the cell automatically opens `kagglehub.login()` so you can paste the token interactively. Never publish the token inside a notebook cell.


In [2]:
import kagglehub

# Download latest version. KaggleHub automatically checks the Colab secret
# named KAGGLE_API_TOKEN. If it is missing, the login widget opens once.
try:
    path = kagglehub.competition_download('m5-forecasting-accuracy')
except Exception as error:
    if error.__class__.__name__ != 'UnauthenticatedError':
        raise
    print('Kaggle authentication is required. Paste your Kaggle API token in the login form below.')
    kagglehub.login()
    path = kagglehub.competition_download('m5-forecasting-accuracy')

print("Path to competition files:", path)

DATA_DIR = Path(path)
if not (DATA_DIR / 'calendar.csv').exists():
    DATA_DIR = next(p.parent for p in DATA_DIR.rglob('calendar.csv'))

required = {'calendar.csv', 'sell_prices.csv', 'sales_train_evaluation.csv'}
available = {p.name for p in DATA_DIR.glob('*.csv')}
assert required.issubset(available), f"Missing files: {sorted(required - available)}"


Official Kaggle M5 competition files loaded successfully for this public snapshot.


## 3. Load and audit the source tables

In [3]:
sales = pd.read_csv(DATA_DIR / 'sales_train_evaluation.csv')
calendar = pd.read_csv(DATA_DIR / 'calendar.csv', parse_dates=['date'])
prices = pd.read_csv(DATA_DIR / 'sell_prices.csv')
day_cols = [c for c in sales.columns if c.startswith('d_')]

audit = pd.DataFrame({
    'table': ['sales', 'calendar', 'prices'],
    'rows': [len(sales), len(calendar), len(prices)],
    'columns': [sales.shape[1], calendar.shape[1], prices.shape[1]],
    'duplicate_business_keys': [
        sales['id'].duplicated().sum(),
        calendar['d'].duplicated().sum(),
        prices.duplicated(['store_id', 'item_id', 'wm_yr_wk']).sum(),
    ],
})
display(audit)
print(f"Products: {sales.item_id.nunique():,} | Stores: {sales.store_id.nunique()} | Historical days: {len(day_cols):,}")


,table,rows,columns,duplicate_business_keys
0,sales,30490,1947,0
1,calendar,1969,14,0
2,prices,6841121,4,0


Products: 3,049 | Stores: 10 | Historical days: 1,941


## 4. Integrate sales, calendar and prices
Sales uses day keys (`d_1`, `d_2`, …). Calendar translates each day into a date and retail week. Price then joins on store, product and retail week. The 100-product preview makes the full relationship easy to inspect.

In [4]:
id_cols = ['item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']
integrated_preview = (
    sales.loc[:99, id_cols + day_cols]
    .melt(id_vars=id_cols, var_name='d', value_name='units')
    .merge(calendar, on='d', how='left', validate='many_to_one')
    .merge(prices, on=['store_id', 'item_id', 'wm_yr_wk'], how='left', validate='many_to_one')
)
integrated_preview['estimated_revenue'] = integrated_preview.units * integrated_preview.sell_price
display(integrated_preview[['date', 'store_id', 'item_id', 'cat_id', 'units', 'sell_price', 'estimated_revenue']].head(10))


,date,store_id,item_id,cat_id,units,sell_price,estimated_revenue
0,2011-01-29,CA_1,HOBBIES_1_001,HOBBIES,0,NaN,NaN
1,2011-01-29,CA_1,HOBBIES_1_002,HOBBIES,0,NaN,NaN
2,2011-01-29,CA_1,HOBBIES_1_003,HOBBIES,0,NaN,NaN
3,2011-01-29,CA_1,HOBBIES_1_004,HOBBIES,0,NaN,NaN
4,2011-01-29,CA_1,HOBBIES_1_005,HOBBIES,0,NaN,NaN
5,2011-01-29,CA_1,HOBBIES_1_006,HOBBIES,0,NaN,NaN
6,2011-01-29,CA_1,HOBBIES_1_007,HOBBIES,0,NaN,NaN
7,2011-01-29,CA_1,HOBBIES_1_008,HOBBIES,12,0.46,5.52
8,2011-01-29,CA_1,HOBBIES_1_009,HOBBIES,2,1.56,3.12
9,2011-01-29,CA_1,HOBBIES_1_010,HOBBIES,0,3.17,0.00


## 5. Five-year demand and seasonality

In [5]:
values = sales[day_cols].to_numpy(dtype=np.float32)
calendar_days = calendar.set_index('d').loc[day_cols].reset_index()
daily = calendar_days[['date', 'year', 'month', 'weekday', 'event_name_1', 'event_type_1']].copy()
daily['units'] = values.sum(axis=0)
daily['moving_average_28'] = daily.units.rolling(28, min_periods=1).mean()
daily['year_week'] = daily.date.dt.to_period('W').astype(str)

fig = go.Figure([
    go.Scatter(x=daily.date, y=daily.units, name='Daily units', opacity=.30),
    go.Scatter(x=daily.date, y=daily.moving_average_28, name='28-day average', line={'width': 3}),
])
fig.update_layout(title='Five years of demand history', yaxis_title='Units')
fig.show()

store_units = pd.DataFrame({
    store: values[sales.store_id.eq(store)].sum(axis=0).sum()
    for store in sorted(sales.store_id.unique())
}, index=['units']).T.reset_index(names='store_id')
px.bar(store_units.sort_values('units'), x='units', y='store_id', orientation='h', title='Total units by store').show()


## 6. Leakage-free 28-day forecast

In [6]:
def score(actual, predicted, history):
    error = actual - predicted
    scale = np.mean(np.diff(history, axis=1) ** 2, axis=1)
    usable = scale > 0
    denominator = np.abs(actual).sum()
    return {
        'MAE': float(np.abs(error).mean()),
        'WAPE': float(np.abs(error).sum() / denominator),
        'RMSSE': float(np.sqrt(np.mean(error[usable] ** 2, axis=1) / scale[usable]).mean()),
        'Bias': float(error.sum() / denominator),
    }


In [7]:
HORIZON = 28
TRAIN_END = len(day_cols) - HORIZON
train = values[:, :TRAIN_END]
actual = values[:, TRAIN_END:]

forecasts = {
    'Last value': np.repeat(train[:, -1:], HORIZON, axis=1),
    'Mean of last 28 days': np.repeat(train[:, -28:].mean(axis=1, keepdims=True), HORIZON, axis=1),
    'Seasonal lag 7': np.tile(train[:, -7:], (1, 4)),
    'Seasonal lag 28': train[:, -28:].copy(),
}
metrics = pd.DataFrame([
    {'model': name, **score(actual, prediction, train)}
    for name, prediction in forecasts.items()
]).sort_values('WAPE')
display(metrics.style.format({'MAE': '{:.4f}', 'WAPE': '{:.2%}', 'RMSSE': '{:.4f}', 'Bias': '{:.2%}'}))
px.bar(metrics.sort_values('WAPE', ascending=False), x='WAPE', y='model', orientation='h', text_auto='.1%', title='28-day baseline comparison').show()


,model,MAE,WAPE,RMSSE,Bias
1,Mean of last 28 days,1.0657,73.86%,0.9240,3.91%
2,Seasonal lag 7,1.2440,86.22%,1.2010,7.36%
3,Seasonal lag 28,1.2840,89.00%,1.2445,3.91%
0,Last value,1.3730,95.16%,1.2063,-13.19%


## 7. Translate the forecast into inventory decisions

In [8]:
best_name = metrics.iloc[0].model
best_forecast = forecasts[best_name]
residual_std = (train[:, -84:] - np.repeat(train[:, -112:-84].mean(axis=1, keepdims=True), 84, axis=1)).std(axis=1)
safety_stock = 1.65 * residual_std * np.sqrt(7)

inventory = pd.DataFrame({
    'item_id': sales.item_id,
    'store_id': sales.store_id,
    'actual_units': actual.sum(axis=1),
    'forecast_units': best_forecast.sum(axis=1),
    'safety_stock': safety_stock,
    'reorder_point': best_forecast[:, :7].sum(axis=1) + safety_stock,
})
inventory['stockout_units'] = (inventory.actual_units - inventory.forecast_units).clip(lower=0)
inventory['excess_units'] = (inventory.forecast_units - inventory.actual_units).clip(lower=0)
display(inventory.head())
display(inventory[['actual_units', 'forecast_units', 'safety_stock', 'stockout_units', 'excess_units']].sum().to_frame('units'))


,item_id,store_id,actual_units,forecast_units,safety_stock,reorder_point,stockout_units,excess_units
0,HOBBIES_1_001,CA_1,33.0,26.999996,4.810535,11.560536,6.000004,0.000000
1,HOBBIES_1_002,CA_1,7.0,2.000000,1.714226,2.214226,5.000000,0.000000
2,HOBBIES_1_003,CA_1,21.0,16.000000,3.657160,7.657160,5.000000,0.000000
3,HOBBIES_1_004,CA_1,49.0,51.000008,8.164436,20.914436,0.000000,2.000008
4,HOBBIES_1_005,CA_1,39.0,38.000000,5.251554,14.751553,1.000000,0.000000


,units
actual_units,1.231764e+06
forecast_units,1.183626e+06
safety_stock,1.774577e+05
stockout_units,1.967820e+05
excess_units,1.486440e+05


## Executive conclusion
This workflow starts from official raw data, validates relational keys, exposes the joins, preserves a future holdout and connects forecast error to stockout and excess-unit decisions. The specialized notebooks below develop each stage in greater depth.